In [2]:
library(tidyverse)
library(vroom)
library(data.table)
library(future)
library(future.apply)

# if pairadise is not installed, install it
if(!requireNamespace("PAIRADISE", quietly = TRUE)) {
  BiocManager::install("PAIRADISE")
}
# Also for BiocParallel
if(!requireNamespace("BiocParallel", quietly = TRUE)) {
  BiocManager::install("BiocParallel")
}
library(PAIRADISE)

reverse_complement <- function(dna_seq) {
  complement <- c("A" = "T", "T" = "A", "C" = "G", "G" = "C")
  nucleotides <- unlist(strsplit(dna_seq, ""))
  complement_nucleotides <- complement[nucleotides]
  reverse_complement_seq <- paste(rev(complement_nucleotides), collapse = "")
  return(reverse_complement_seq)
}

args <- commandArgs(trailingOnly = TRUE)
celltype1 <- "A375"
celltype2 <- "all"
output_filename <- args[3]

print(paste("Celltype 1: ", celltype1))
print(paste("Celltype 2: ", celltype2))
print(paste("Output filename: ", output_filename))
all_sample_reps <- fread("/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4_merged/WT_all_samples_PSI_count_table.csv")
all_sample_reps <- all_sample_reps %>% 
  mutate(index_offset = paste0(index, "__", offset)) 
print("Finished reading data")


[1] "Celltype 1:  A375"
[1] "Celltype 2:  all"
[1] "Output filename:  NA"


[1] "Finished reading data"


In [3]:
all_sample_reps

index,mode,offset,sample,condition,included_count,skipped_count,index_offset
<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<chr>
ENSG00000000003.15;TSPAN6;chrX-100632484-100632568-100630758-100630866-100633404-100633539,INCLUDED,0:-69:0,769P-rep1,769P,0,0,ENSG00000000003.15;TSPAN6;chrX-100632484-100632568-100630758-100630866-100633404-100633539__0:-69:0
ENSG00000000003.15;TSPAN6;chrX-100632484-100632568-100630758-100630866-100633404-100633539,INCLUDED,0:0:0,769P-rep1,769P,29,0,ENSG00000000003.15;TSPAN6;chrX-100632484-100632568-100630758-100630866-100633404-100633539__0:0:0
ENSG00000000003.15;TSPAN6;chrX-100632484-100632568-100630758-100630866-100633404-100633539,INCLUDED,15:0:0,769P-rep1,769P,0,0,ENSG00000000003.15;TSPAN6;chrX-100632484-100632568-100630758-100630866-100633404-100633539__15:0:0
ENSG00000000003.15;TSPAN6;chrX-100633930-100634029-100632484-100632568-100635177-100635252,INCLUDED,0:0:0,769P-rep1,769P,87,1,ENSG00000000003.15;TSPAN6;chrX-100633930-100634029-100632484-100632568-100635177-100635252__0:0:0
ENSG00000000003.15;TSPAN6;chrX-100633930-100634029-100632484-100632568-100635177-100635252,INCLUDED,11:0:0,769P-rep1,769P,0,1,ENSG00000000003.15;TSPAN6;chrX-100633930-100634029-100632484-100632568-100635177-100635252__11:0:0
ENSG00000000003.15;TSPAN6;chrX-100635177-100635252-100633930-100634029-100635557-100635746,INCLUDED,0:0:0,769P-rep1,769P,6,0,ENSG00000000003.15;TSPAN6;chrX-100635177-100635252-100633930-100634029-100635557-100635746__0:0:0
ENSG00000000419.14;DPM1;chr20-50945736-50945762-50942030-50942126-50945846-50945861,INCLUDED,0:0:0,769P-rep1,769P,16,4,ENSG00000000419.14;DPM1;chr20-50945736-50945762-50942030-50942126-50945846-50945861__0:0:0
ENSG00000000419.14;DPM1;chr20-50948628-50948662-50945736-50945923-50955185-50955285,INCLUDED,0:0:0,769P-rep1,769P,0,3,ENSG00000000419.14;DPM1;chr20-50948628-50948662-50945736-50945923-50955185-50955285__0:0:0
ENSG00000000457.14;SCYL3;chr1-169870254-169870357-169868927-169869039-169873695-169873752,INCLUDED,0:-5:0,769P-rep1,769P,0,9,ENSG00000000457.14;SCYL3;chr1-169870254-169870357-169868927-169869039-169873695-169873752__0:-5:0


In [8]:
df <- all_sample_reps
  condition1 <- "A375"
  condition2 <- "all"
  
  # Filter data frames based on conditions
  condition1_df <- df %>% filter(condition == condition1) %>% mutate(PSI = included_count / (included_count + skipped_count))
  if (condition2 == "all") {
    condition2_df <- df %>% filter(condition != condition1) %>% mutate(PSI = included_count / (included_count + skipped_count))
  } else {
    condition2_df <- df %>% filter(condition == condition2) %>% mutate(PSI = included_count / (included_count + skipped_count))
  }
  
  condition1_df_PSI <- condition1_df %>% group_by(index_offset) %>% 
    summarise(PSI_1 = mean(PSI,na.rm=T))

  condition2_df_PSI <- condition2_df %>% group_by(index_offset) %>% 
    summarise(PSI_2 = mean(PSI, na.rm = T))

  # Merge the 2 dataframe. 
  condition1_2_merged <- merge(condition1_df_PSI, condition2_df_PSI, by = "index_offset")  %>% 
                            mutate(PSI_diff = abs(PSI_1 - PSI_2)) %>% filter(PSI_diff >= 0.1)

  # Create a list of all unique indices.
  unique_indices <- unique(condition1_2_merged$index_offset)
  
  # Function to pad vectors with zeros
  pad_with_zeros <- function(vec, target_len = 3) {
    if (length(vec) < target_len) {
      vec <- c(vec, rep(0, target_len - length(vec)))
    }
    return(vec)
  }

[1] 23964

# Cell line data with transcriptomic groups. 

In [ ]:
library(tidyverse)
library(vroom)
library(data.table)
library(future)
library(future.apply)

# if pairadise is not installed, install it
if(!requireNamespace("PAIRADISE", quietly = TRUE)) {
  BiocManager::install("PAIRADISE")
}
# Also for BiocParallel
if(!requireNamespace("BiocParallel", quietly = TRUE)) {
  BiocManager::install("BiocParallel")
}
library(PAIRADISE)

reverse_complement <- function(dna_seq) {
  complement <- c("A" = "T", "T" = "A", "C" = "G", "G" = "C")
  nucleotides <- unlist(strsplit(dna_seq, ""))
  complement_nucleotides <- complement[nucleotides]
  reverse_complement_seq <- paste(rev(complement_nucleotides), collapse = "")
  return(reverse_complement_seq)
}

args <- commandArgs(trailingOnly = TRUE)
cluster_number <- args[1]
output_filename <- args[3]

print(paste("Celltype 1: ", celltype1))
print(paste("Celltype 2: ", celltype2))
print(paste("Output filename: ", output_filename))
all_sample_reps <- fread("/broad/dawnccle/processed_data/reprocess_250221/count_normalized_v4_merged/WT_all_samples_PSI_count_table.csv")
all_sample_reps <- all_sample_reps %>% 
  mutate(index_offset = paste0(index, "__", offset)) 
print("Finished reading data")

transcriptomic_groups <- fread("/broad/dawnccle/melange/data/cellline_data_with_cluster.csv")


get_paradise_output <- function(df, cluster_number) {
  target_conditions <- transcriptomic_groups %>% filter(cluster == cluster_number) %>% pull(display_name)
  non_target_conditions <- transcriptomic_groups %>% filter(cluster != cluster_number) %>% pull(display_name)

  # Filter data frames based on conditions
  condition1_df <- df %>% filter(condition %in% target_conditions) %>% mutate(PSI = included_count / (included_count + skipped_count))
  condition2_df <- df %>% filter(condition %in% non_target_conditions) %>% mutate(PSI = included_count / (included_count + skipped_count))
  
  condition1_df_PSI <- condition1_df %>% group_by(index_offset) %>% 
    summarise(PSI_1 = mean(PSI,na.rm=T))

  condition2_df_PSI <- condition2_df %>% group_by(index_offset) %>% 
    summarise(PSI_2 = mean(PSI, na.rm = T))

  # Merge the 2 dataframe.
  # Also filter out the indices with PSI difference less than 0.1 so that we don't have to run calculations for them.
  condition1_2_merged <- merge(condition1_df_PSI, condition2_df_PSI, by = "index_offset")  %>% 
                            mutate(PSI_diff = abs(PSI_1 - PSI_2)) %>% filter(PSI_diff >= 0.1)

  # Create a list of all unique indices.
  unique_indices <- unique(condition1_2_merged$index_offset)
  
  # Function to pad vectors with zeros
  pad_with_zeros <- function(vec, target_len = 3) {
    if (length(vec) < target_len) {
      vec <- c(vec, rep(0, target_len - length(vec)))
    }
    return(vec)
  }
  
  
  # Create the paradise data frame using dplyr operations and future.apply for parallel processing
  paradise_df <- lapply(unique_indices, function(idx) {
    condition1_index_df <- condition1_df %>% filter(index_offset == idx) %>% arrange(sample)
    condition2_index_df <- condition2_df %>% filter(index_offset == idx) %>% arrange(sample)
    
    I1 <- pad_with_zeros(condition1_index_df$included_count)
    S1 <- pad_with_zeros(condition1_index_df$skipped_count)
    I2 <- pad_with_zeros(condition2_index_df$included_count)
    S2 <- pad_with_zeros(condition2_index_df$skipped_count)
    
    return (data.frame(
      ExonID = idx, 
      I1 = paste(I1, collapse = ","), 
      S1 = paste(S1, collapse = ","), 
      I2 = paste(I2, collapse = ","), 
      S2 = paste(S2, collapse = ","), 
      I_len = 1, 
      S_len = 1
    ))
  }) %>% bind_rows()
  
  return(paradise_df)
}

formatted_df <- get_paradise_output(all_sample_reps, cluster_number)
print("Finished formatting data")
print(output_filename)
write_tsv(formatted_df, file = output_filename)


In [21]:
transcriptomic_groups <- fread("/mnt/dawnccle2/melange/data/cellline_data_with_cluster.csv")

cluster_number <- 1

# Get the cell lines in this cluster
cell_lines <- transcriptomic_groups %>% filter(cluster == cluster_number) %>% pull(display_name)


unique(transcriptomic_groups$cluster)
# Write this to a file as a df. Only 1 column, no header.
# write_tsv(data.frame(cluster = unique(transcriptomic_groups$cluster)), file = "/mnt/dawnccle2/melange/data/transcriptomic_groups.txt", col_names = FALSE)
cell_lines

[1]  1 39 45  2  5 18 34 23 20 49 19 28 29 22  8

[1] "697"   "SEM"   "NALM6" "KOPN8"

In [1]:
library(tidyverse)
library(vroom)
library(data.table)
library(future)
library(future.apply)

# if pairadise is not installed, install it
if(!requireNamespace("PAIRADISE", quietly = TRUE)) {
  BiocManager::install("PAIRADISE")
}
# Also for BiocParallel
if(!requireNamespace("BiocParallel", quietly = TRUE)) {
  BiocManager::install("BiocParallel")
}
library(PAIRADISE)

reverse_complement <- function(dna_seq) {
  complement <- c("A" = "T", "T" = "A", "C" = "G", "G" = "C")
  nucleotides <- unlist(strsplit(dna_seq, ""))
  complement_nucleotides <- complement[nucleotides]
  reverse_complement_seq <- paste(rev(complement_nucleotides), collapse = "")
  return(reverse_complement_seq)
}

args <- commandArgs(trailingOnly = TRUE)
celltype1 <- "A375"
celltype2 <- "all"
output_filename <- args[3]

print(paste("Celltype 1: ", celltype1))
print(paste("Celltype 2: ", celltype2))
print(paste("Output filename: ", output_filename))
all_sample_reps <- fread("/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4_merged/WT_all_samples_raw_counts.csv")
all_sample_reps <- all_sample_reps %>% 
  filter(mode == "INCLUDED") %>% 
  mutate(index_offset = paste0(index, "__", offset)) 


all_event_pairs <- fread("/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4_merged/WT_included_all_unique_event_pairs.csv") %>% select(-index)
print("Finished reading data")


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘vroom’


The following objects are masked from ‘package:readr’:

    as.col_spec, col_character, col_date, col_datetime, col_double,
    col_factor, col_guess, col_integer, col_logical, col_number,
    col_skip, col_time, cols, cols_condense, cols_only, date_names,
    date_names_lang, date_names_langs, default_locale, fwf_cols,
    fwf_empty, fwf_positions, fwf_widths, locale, output_column,
    problems, spec



Attaching package: ‘data.table’


The

[1] "Celltype 1:  A375"
[1] "Celltype 2:  all"
[1] "Output filename:  NA"
[1] "Finished reading data"


In [3]:
df <- all_sample_reps
all_event_pairs <- all_event_pairs %>% mutate(index_offset_major_minor = paste0(index_offset_major, "___", index_offset_minor))
condition1 <- "A375"
condition2 <- "all"

# Create a function to process samples to avoid code duplication
process_samples <- function(df, all_event_pairs, samples) {
  # Pre-filter the data frames to avoid repeated filtering in the loop
  major_indices <- all_event_pairs$index_offset_major
  minor_indices <- all_event_pairs$index_offset_minor
  valid_major_minor_pairs <- all_event_pairs$index_offset_major_minor
  
  # Get data for this samples
  sample_df <- df %>% filter(sample %in% samples)
  
  # Filter once for major and minor indices
  sample_df_major <- sample_df %>% 
    filter(index_offset %in% major_indices) %>% 
    select(-count)
  
  sample_df_minor <- sample_df %>% 
    filter(index_offset %in% minor_indices) %>% 
    select(-count, -condition)
  
  # Join and filter
  sample_df_with_counts <- sample_df_major %>% 
    left_join(sample_df_minor, by = c("index" = "index", "sample" = "sample"), relationship = "many-to-many") %>% 
    mutate(index_offset_major_minor = paste0(index_offset.x, "___", index_offset.y)) %>% 
    filter(index_offset_major_minor %in% valid_major_minor_pairs)
  
  return(sample_df_with_counts)
}

# Get unique samples for each condition
unique_samples_condition1 <- df %>% 
  filter(condition == condition1) %>% 
  pull(sample) %>% 
  unique()

if (condition2 == "all") {
  unique_samples_condition2 <- df %>% 
    filter(condition != condition1) %>% 
    pull(sample) %>% 
    unique()
} else {
  unique_samples_condition2 <- df %>% 
    filter(condition == condition2) %>% 
    pull(sample) %>% 
    unique()
}

# Process each condition using the function
condition1_df <- process_samples(df, all_event_pairs, unique_samples_condition1)
condition2_df <- process_samples(df, all_event_pairs, unique_samples_condition2)


In [18]:
condition1_df_summary <- condition1_df %>% 
  mutate(altSS_1 = count_scaled.x/(count_scaled.x + count_scaled.y)) %>% 
  group_by(index_offset_major_minor) %>% 
  summarise(altSS_1 = mean(altSS_1, na.rm = TRUE)) %>% 
  select(index_offset_major_minor, altSS_1)

condition2_df_summary <- condition2_df %>% 
    mutate(altSS_2 = (count_scaled.x)/(count_scaled.x + count_scaled.y)) %>% 
    group_by(index_offset_major_minor) %>% 
    summarise(altSS_2 = mean(altSS_2, na.rm = TRUE)) %>% 
    select(index_offset_major_minor, altSS_2)

condition1_2_merged <- merge(condition1_df_summary, condition2_df_summary, by = c("index_offset_major_minor")) %>% 
  mutate(altSS_diff = abs(altSS_1 - altSS_2)) %>%
  filter(altSS_diff >= 0.1)


In [27]:
# Create a function to process samples to avoid code duplication
process_samples <- function(df, all_event_pairs, samples) {
  # Pre-filter the data frames to avoid repeated filtering in the loop
  major_indices <- all_event_pairs$index_offset_major
  minor_indices <- all_event_pairs$index_offset_minor
  valid_major_minor_pairs <- all_event_pairs$index_offset_major_minor
  
  # Get data for this samples
  sample_df <- df %>% filter(sample %in% samples)
  
  # Filter once for major and minor indices
  sample_df_major <- sample_df %>% 
    filter(index_offset %in% major_indices) %>% 
    select(-count)
  
  sample_df_minor <- sample_df %>% 
    filter(index_offset %in% minor_indices) %>% 
    select(-count, -condition)
  
  # Join and filter
  sample_df_with_counts <- sample_df_major %>% 
    left_join(sample_df_minor, by = c("index" = "index", "sample" = "sample"), relationship = "many-to-many") %>% 
    mutate(index_offset_major_minor = paste0(index_offset.x, "___", index_offset.y)) %>% 
    filter(index_offset_major_minor %in% valid_major_minor_pairs)
  
  return(sample_df_with_counts)
}

get_paradise_output <- function(df, all_event_pairs, condition1, condition2) {
# Get unique samples for each condition
unique_samples_condition1 <- df %>% 
  filter(condition == condition1) %>% 
  pull(sample) %>% 
  unique()

if (condition2 == "all") {
  unique_samples_condition2 <- df %>% 
    filter(condition != condition1) %>% 
    pull(sample) %>% 
    unique()
} else {
  unique_samples_condition2 <- df %>% 
    filter(condition == condition2) %>% 
    pull(sample) %>% 
    unique()
}

# Process each condition using the function
condition1_df <- process_samples(df, all_event_pairs, unique_samples_condition1)
condition2_df <- process_samples(df, all_event_pairs, unique_samples_condition2)
  
condition1_df_summary <- condition1_df %>% 
  mutate(altSS_1 = count_scaled.x/(count_scaled.x + count_scaled.y)) %>% 
  group_by(index_offset_major_minor) %>% 
  summarise(altSS_1 = mean(altSS_1, na.rm = TRUE)) %>% 
  select(index_offset_major_minor, altSS_1)

condition2_df_summary <- condition2_df %>% 
    mutate(altSS_2 = (count_scaled.x)/(count_scaled.x + count_scaled.y)) %>% 
    group_by(index_offset_major_minor) %>% 
    summarise(altSS_2 = mean(altSS_2, na.rm = TRUE)) %>% 
    select(index_offset_major_minor, altSS_2)

condition1_2_merged <- merge(condition1_df_summary, condition2_df_summary, by = c("index_offset_major_minor")) %>% 
  mutate(altSS_diff = abs(altSS_1 - altSS_2)) %>%
  filter(altSS_diff >= 0.1)


  # Create a list of all unique indices.
  unique_indices <- unique(condition1_2_merged$index_offset_major_minor)
  
  # Function to pad vectors with zeros
  pad_with_zeros <- function(vec, target_len = 3) {
    if (length(vec) < target_len) {
      vec <- c(vec, rep(0, target_len - length(vec)))
    }
    return(vec)
  }
  
  
  # Create the paradise data frame using dplyr operations and future.apply for parallel processing
  paradise_df <- lapply(unique_indices, function(idx) {
    condition1_index_df <- condition1_df %>% filter(index_offset_major_minor == idx) %>% arrange(sample)
    condition2_index_df <- condition2_df %>% filter(index_offset_major_minor == idx) %>% arrange(sample)
    
    I1 <- pad_with_zeros(condition1_index_df$count_scaled.x)
    S1 <- pad_with_zeros(condition1_index_df$count_scaled.y)
    I2 <- pad_with_zeros(condition2_index_df$count_scaled.x)
    S2 <- pad_with_zeros(condition2_index_df$count_scaled.y)
    
    return (data.frame(
      ExonID = idx, 
      I1 = paste(I1, collapse = ","), 
      S1 = paste(S1, collapse = ","), 
      I2 = paste(I2, collapse = ","), 
      S2 = paste(S2, collapse = ","), 
      I_len = 1, 
      S_len = 1
    ))
  }) %>% bind_rows()
  
  return(paradise_df)
}

In [28]:
formatted_df <- get_paradise_output(all_sample_reps, all_event_pairs, celltype1, celltype2)

In [7]:
library(tidyverse)
library(vroom)
library(data.table)
library(future)
library(future.apply)

# if pairadise is not installed, install it
if (!requireNamespace("PAIRADISE", quietly = TRUE)) {
  BiocManager::install("PAIRADISE")
}
# Also for BiocParallel
if (!requireNamespace("BiocParallel", quietly = TRUE)) {
  BiocManager::install("BiocParallel")
}
library(PAIRADISE)

reverse_complement <- function(dna_seq) {
  complement <- c("A" = "T", "T" = "A", "C" = "G", "G" = "C")
  nucleotides <- unlist(strsplit(dna_seq, ""))
  complement_nucleotides <- complement[nucleotides]
  reverse_complement_seq <- paste(rev(complement_nucleotides), collapse = "")
  return(reverse_complement_seq)
}

args <- commandArgs(trailingOnly = TRUE)
celltype1 <- "K562WT"
celltype2 <- "K562K700E"
output_filename <- args[3]

print(paste("Celltype 1: ", celltype1))
print(paste("Celltype 2: ", celltype2))
print(paste("Output filename: ", output_filename))
all_sample_reps <- fread("/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4_merged/WT_all_samples_raw_counts.csv")
all_sample_reps <- all_sample_reps %>% 
  filter(mode == "INCLUDED") %>% 
  filter(grepl("DLST", index)) %>%
  mutate(index_offset = paste0(index, "__", offset)) 


all_event_pairs <- fread("/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4_merged/WT_included_all_unique_event_pairs.csv") %>% 
  select(-index) %>% 
  mutate(index_offset_major_minor = paste0(index_offset_major, "___", index_offset_minor))
print("Finished reading data")




[1] "Celltype 1:  K562WT"
[1] "Celltype 2:  K562K700E"
[1] "Output filename:  NA"


[1] "Finished reading data"


In [9]:
# Create a function to process samples to avoid code duplication
process_samples <- function(df, all_event_pairs, samples) {
  # Pre-filter the data frames to avoid repeated filtering in the loop
  major_indices <- all_event_pairs$index_offset_major
  minor_indices <- all_event_pairs$index_offset_minor
  valid_major_minor_pairs <- all_event_pairs$index_offset_major_minor
  
  sample_df <- df %>% filter(sample %in% samples) %>% select(-count)
  unique_indices_in_sample <- unique(sample_df$index_offset)
  # Create a table of all combinations of valid_major_minor_pairs and samples
  # First, get unique samples
  unique_samples <- unique(samples)
  print("Unique samples")
  print(unique_samples)
  print("Unique major indices")
  print(length(major_indices))
  print("Unique minor indices")
  print(length(minor_indices))
  print("Unique indices in sample")
  print(length(unique_indices_in_sample))
  # Create a data frame with all combinations
  combinations_table <- expand.grid(
    index_offset = unique(unique_indices_in_sample),
    sample = unique_samples,
    stringsAsFactors = FALSE
  )
  print("Combinations table")
  print(nrow(combinations_table))

  # Merge the combinations table with the sample data
  sample_df <- merge(combinations_table, sample_df, by = c("index_offset", "sample"), all.x = TRUE)
  # Set all count_scaled to 0 if they are NA
  sample_df$count_scaled <- ifelse(is.na(sample_df$count_scaled), 0, sample_df$count_scaled)
  # Reassign index by splitting index_offset
  sample_df <- sample_df %>%
    separate(index_offset, into = c("index", "offset"), sep = "__", remove = FALSE)
  print("Sample df")
  print(nrow(sample_df))
  print(sample_df)
  sample_df_major <- sample_df %>% 
    filter(index_offset %in% major_indices)
  sample_df_minor <- sample_df %>% 
      select(-condition) %>% 
      filter(index_offset %in% minor_indices)


  # Join and filter. 
  sample_df_with_counts <- sample_df_major %>% 
    left_join(sample_df_minor, by = c("index" = "index", "sample" = "sample"), relationship = "many-to-many") %>% 
    mutate(count_scaled.x = ifelse(is.na(count_scaled.x), 0, count_scaled.x),
           count_scaled.y = ifelse(is.na(count_scaled.y), 0, count_scaled.y),
           index_offset_major_minor = paste0(index_offset.x, "___", index_offset.y)) %>% 
    filter(index_offset_major_minor %in% valid_major_minor_pairs)
  
  return(sample_df_with_counts)
}

get_paradise_output <- function(df, all_event_pairs, condition1, condition2) {
  # Get unique samples for each condition
  unique_samples_condition1 <- df %>% 
    filter(condition == condition1) %>% 
    pull(sample) %>% 
    unique()
  
  if (condition2 == "all") {
    unique_samples_condition2 <- df %>% 
      filter(condition != condition1) %>% 
      pull(sample) %>% 
      unique()
  } else {
    unique_samples_condition2 <- df %>% 
      filter(condition == condition2) %>% 
      pull(sample) %>% 
      unique()
  }
  
  # Process each condition using the function
  condition1_df <- process_samples(df, all_event_pairs, unique_samples_condition1)
  condition2_df <- process_samples(df, all_event_pairs, unique_samples_condition2)
  
  condition1_df_summary <- condition1_df %>% 
    mutate(altSS_1 = count_scaled.x/(count_scaled.x + count_scaled.y)) %>% 
    group_by(index_offset_major_minor) %>% 
    summarise(altSS_1 = mean(altSS_1, na.rm = TRUE)) %>% 
    select(index_offset_major_minor, altSS_1)
  
  condition2_df_summary <- condition2_df %>% 
    mutate(altSS_2 = (count_scaled.x)/(count_scaled.x + count_scaled.y)) %>% 
    group_by(index_offset_major_minor) %>% 
    summarise(altSS_2 = mean(altSS_2, na.rm = TRUE)) %>% 
    select(index_offset_major_minor, altSS_2)
  
  condition1_2_merged <- merge(condition1_df_summary, condition2_df_summary, by = c("index_offset_major_minor")) %>% 
    mutate(altSS_diff = abs(altSS_1 - altSS_2)) 
  
  
  # Create a list of all unique indices.
  unique_indices <- unique(condition1_2_merged$index_offset_major_minor)
  
  # Function to pad vectors with zeros
  pad_with_zeros <- function(vec, target_len = 3) {
    if (length(vec) < target_len) {
      vec <- c(vec, rep(0, target_len - length(vec)))
    }
    return(vec)
  }
  
  
  # Create the paradise data frame using dplyr operations and future.apply for parallel processing
  paradise_df <- lapply(unique_indices, function(idx) {
    condition1_index_df <- condition1_df %>% filter(index_offset_major_minor == idx) %>% arrange(sample)
    condition2_index_df <- condition2_df %>% filter(index_offset_major_minor == idx) %>% arrange(sample)
    
    I1 <- pad_with_zeros(condition1_index_df$count_scaled.x)
    S1 <- pad_with_zeros(condition1_index_df$count_scaled.y)
    I2 <- pad_with_zeros(condition2_index_df$count_scaled.x)
    S2 <- pad_with_zeros(condition2_index_df$count_scaled.y)
    
    return(data.frame(
      ExonID = idx, 
      I1 = paste(I1, collapse = ","), 
      S1 = paste(S1, collapse = ","), 
      I2 = paste(I2, collapse = ","), 
      S2 = paste(S2, collapse = ","), 
      I_len = 1, 
      S_len = 1
    ))
  }) %>% bind_rows()
  
  return(paradise_df)
}

formatted_df <- get_paradise_output(all_sample_reps, all_event_pairs, celltype1, celltype2)
formatted_df

[1] "Unique samples"
[1] "K562WT-rep1" "K562WT-rep2" "K562WT-rep3"
[1] "Unique major indices"
[1] 414785
[1] "Unique minor indices"
[1] 414785
[1] "Unique indices in sample"
[1] 8
[1] "Combinations table"
[1] 24
[1] "Sample df"
[1] 24
                                                                                   index_offset
1    ENSG00000119689.15;DLST;chr14-74885585-74885634-74881902-74882016-74889094-74889147__0:0:0
2    ENSG00000119689.15;DLST;chr14-74885585-74885634-74881902-74882016-74889094-74889147__0:0:0
3    ENSG00000119689.15;DLST;chr14-74885585-74885634-74881902-74882016-74889094-74889147__0:0:0
4   ENSG00000119689.15;DLST;chr14-74889276-74889349-74881915-74882016-74891055-74891167__-2:0:0
5   ENSG00000119689.15;DLST;chr14-74889276-74889349-74881915-74882016-74891055-74891167__-2:0:0
6   ENSG00000119689.15;DLST;chr14-74889276-74889349-74881915-74882016-74891055-74891167__-2:0:0
7    ENSG00000119689.15;DLST;chr14-74889276-74889349-74881915-74882016-74891055-74891167__0:0

ExonID,I1,S1,I2,S2,I_len,S_len
<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>
ENSG00000119689.15;DLST;chr14-74889276-74889349-74881915-74882016-74891055-74891167__-2:0:0___ENSG00000119689.15;DLST;chr14-74889276-74889349-74881915-74882016-74891055-74891167__0:0:0,"56,132,69","5,9,2","84,81,37","1,0,0",1,1
ENSG00000119689.15;DLST;chr14-74889896-74889952-74889274-74889349-74891055-74891167__0:0:0___ENSG00000119689.15;DLST;chr14-74889896-74889952-74889274-74889349-74891055-74891167__-19:0:0,"114,94,78","0,1,4","33,41,31","113,151,110",1,1


In [38]:
  combinations_table <- expand.grid(
    index_offset = unique(c(all_event_pairs$index_offset_major, all_event_pairs$index_offset_minor)),
    sample = unique_samples,
    stringsAsFactors = FALSE
  )
  df

function (x, df1, df2, ncp, log = FALSE) 
{
    if (missing(ncp)) 
        .Call(C_df, x, df1, df2, log)
    else .Call(C_dnf, x, df1, df2, ncp, log)
}
<bytecode: 0x5557845d46e8>
<environment: namespace:stats>

In [26]:
formatted_df

ExonID,I1,S1,I2,S2,I_len,S_len
<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>
ENSG00000119689.15;DLST;chr14-74889276-74889349-74881915-74882016-74891055-74891167__-2:0:0___ENSG00000119689.15;DLST;chr14-74889276-74889349-74881915-74882016-74891055-74891167__0:0:0,"56,132,69","5,9,2","84,0,0","1,0,0",1,1
ENSG00000119689.15;DLST;chr14-74889896-74889952-74889274-74889349-74891055-74891167__0:0:0___ENSG00000119689.15;DLST;chr14-74889896-74889952-74889274-74889349-74891055-74891167__-19:0:0,"94,78,0","1,4,0","33,41,31","113,151,110",1,1


In [28]:
all_sample_reps %>% 
  filter(condition %in% c("K562K700E", "K562WT")) %>% 
  filter(offset %in% c("0:0:0", "-2:0:0", "-19:0:0")) 

index,mode,offset,count,count_scaled,sample,condition,index_offset
<chr>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<chr>
ENSG00000119689.15;DLST;chr14-74885585-74885634-74881902-74882016-74889094-74889147,INCLUDED,0:0:0,84,84,K562K700E-rep1,K562K700E,ENSG00000119689.15;DLST;chr14-74885585-74885634-74881902-74882016-74889094-74889147__0:0:0
ENSG00000119689.15;DLST;chr14-74889276-74889349-74881915-74882016-74891055-74891167,INCLUDED,-2:0:0,84,84,K562K700E-rep1,K562K700E,ENSG00000119689.15;DLST;chr14-74889276-74889349-74881915-74882016-74891055-74891167__-2:0:0
ENSG00000119689.15;DLST;chr14-74889276-74889349-74881915-74882016-74891055-74891167,INCLUDED,0:0:0,1,1,K562K700E-rep1,K562K700E,ENSG00000119689.15;DLST;chr14-74889276-74889349-74881915-74882016-74891055-74891167__0:0:0
ENSG00000119689.15;DLST;chr14-74889896-74889952-74889274-74889349-74891055-74891167,INCLUDED,-19:0:0,113,113,K562K700E-rep1,K562K700E,ENSG00000119689.15;DLST;chr14-74889896-74889952-74889274-74889349-74891055-74891167__-19:0:0
ENSG00000119689.15;DLST;chr14-74889896-74889952-74889274-74889349-74891055-74891167,INCLUDED,0:0:0,33,33,K562K700E-rep1,K562K700E,ENSG00000119689.15;DLST;chr14-74889896-74889952-74889274-74889349-74891055-74891167__0:0:0
ENSG00000119689.15;DLST;chr14-74891055-74891167-74889274-74889349-74892833-74892943,INCLUDED,0:0:0,13,13,K562K700E-rep1,K562K700E,ENSG00000119689.15;DLST;chr14-74891055-74891167-74889274-74889349-74892833-74892943__0:0:0
ENSG00000119689.15;DLST;chr14-74894311-74894409-74893347-74893424-74898368-74898455,INCLUDED,0:0:0,1,1,K562K700E-rep1,K562K700E,ENSG00000119689.15;DLST;chr14-74894311-74894409-74893347-74893424-74898368-74898455__0:0:0
ENSG00000119689.15;DLST;chr14-74900288-74900372-74899922-74899996-74901065-74901190,INCLUDED,0:0:0,6,6,K562K700E-rep1,K562K700E,ENSG00000119689.15;DLST;chr14-74900288-74900372-74899922-74899996-74901065-74901190__0:0:0
ENSG00000119689.15;DLST;chr14-74885585-74885634-74881902-74882016-74889094-74889147,INCLUDED,0:0:0,91,91,K562K700E-rep2,K562K700E,ENSG00000119689.15;DLST;chr14-74885585-74885634-74881902-74882016-74889094-74889147__0:0:0
